# 03 — Compare All Controllers

Runs `evaluation/compare_controllers.py` from the cloned repository.
No evaluation code lives in this notebook.

Controllers evaluated under **identical** conditions (same profiles, same seed):

| Controller | Type |
|---|---|
| No cooling | Baseline |
| Constant u=0.5 | Baseline |
| Constant u=1.0 | Baseline |
| Bang-bang | Classical |
| Proportional | Classical |
| PI tuned | Classical |
| PPO | RL |
| SAC | RL |

> Train PPO and SAC first (`01_` and `02_` notebooks) before running this.

## 1 — Mount Drive and Clone / Pull Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/RL-Battery-Thermal-Management.git"
REPO_DIR = "/content/RL-Battery-Thermal-Management"

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

## 2 — Project Root and Dependencies

In [ ]:
%cd /content/RL-Battery-Thermal-Management
!pip install -q -r requirements.txt
print("Ready.")

## 3 — Model and Results Paths

In [ ]:
BASE = "/content/drive/MyDrive/battery_rl"

PPO_MODEL   = f"{BASE}/models/ppo_3d_pack/ppo_pack_final.zip"
PPO_VECNORM = f"{BASE}/models/ppo_3d_pack/vec_normalize.pkl"
SAC_MODEL   = f"{BASE}/models/sac_3d_pack/sac_pack_final.zip"
RESULTS_DIR = f"{BASE}/results/comparison"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Report what exists on Drive
for label, path in [("PPO model",   PPO_MODEL),
                     ("PPO vecnorm", PPO_VECNORM),
                     ("SAC model",   SAC_MODEL)]:
    status = "found  " if os.path.exists(path) else "MISSING"
    print(f"[{status}] {label}: {path}")

## 4 — Diagnostic

In [ ]:
!pwd
!find envs -maxdepth 2 -type f | sort
!find configs -maxdepth 2 -type f | sort
!grep -R "from envs" -n training
!grep -R "from configs" -n training

## 5 — Run Comparison

`--episodes 20` runs 5 rounds of the 4 heat profiles each.
PPO and SAC are skipped gracefully if their model files are not found;
only classical baselines run in that case.

In [ ]:
!PYTHONPATH=. python evaluation/compare_controllers.py \
  --ppo-model   {PPO_MODEL} \
  --ppo-vecnorm {PPO_VECNORM} \
  --sac-model   {SAC_MODEL} \
  --results-dir {RESULTS_DIR} \
  --episodes 20 \
  --seed 7

## 6 — Results Summary

In [ ]:
print("Output files:")
for f in sorted(os.listdir(RESULTS_DIR)):
    size_kb = os.path.getsize(os.path.join(RESULTS_DIR, f)) // 1024
    print(f"  {f}  ({size_kb} KB)")

In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(RESULTS_DIR, "comparison_metrics.csv"))

summary = (
    df.groupby("controller")[["total_reward", "T_max_peak_C",
                               "T_gradient_max_C", "time_above_safe_s",
                               "total_cooling_effort"]]
      .mean()
      .sort_values("total_reward", ascending=False)
      .round(2)
)
print("Mean metrics across all profiles and rounds:\n")
print(summary.to_string())

## 7 — Display Plots

In [ ]:
from IPython.display import Image, display
import glob

plots = sorted(glob.glob(os.path.join(RESULTS_DIR, "*.png")))
print(f"Displaying {len(plots)} plots:")
for p in plots:
    print(" ", os.path.basename(p))
    display(Image(p, width=900))

## Baselines Only (no RL models required)

Useful to benchmark classical controllers before training is done.

In [ ]:
# Uncomment to run classical controllers only:

# BASELINE_RESULTS = f"{BASE}/results/baselines_only"
# os.makedirs(BASELINE_RESULTS, exist_ok=True)
#
# !PYTHONPATH=. python evaluation/compare_controllers.py \
#   --results-dir {BASELINE_RESULTS} \
#   --episodes 4 \
#   --seed 7